# SETU: Hinglish RAG Research Dataset & Benchmark Pipeline
**Author:** Anonymous Authors | **Target:** IEEE / ACM RAG Research

This Google Colab notebook automates the entire end-to-end pipeline on public Hinglish research datasets:
1. **Dataset Download:** Streams public peer-reviewed Hinglish datasets (`IndicVault-Hinglish100k`, `Google Hinglish-TOP`, or `PHINC`).
2. **GPU Encoding:** Fast embedding generation using state-of-the-art multilingual bi-encoders (`BGE-M3` and `mE5-large`).
3. **Zero-Shot Baselines:** Evaluates dense retrieval (MRR, Recall@10, NDCG@10).
4. **The Overcorrection Diagnostic:** Measures whether corrective operators help or degrade retrieval.
5. **The 4-Way Reference Comparison & Oracle Gap:** Evaluates *Never Fix* vs *Always Fix* vs *Confidence Cutoff* vs *Empirical Oracle*.

### Step 1: Install Required Packages & Check GPU

In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu rapidfuzz scikit-learn lightgbm
!nvidia-smi

### Step 2: Download & Format Research Dataset
Choose between **Google Hinglish-TOP** (Human-annotated queries in Latin script) or **Dolly-15k Hinglish** (Romanized QA pairs).

In [ ]:
import json
import re
from collections import Counter
from datasets import load_dataset

DATASET_CHOICE = "hinglish_top"  # Options: 'hinglish_top', 'dolly_hinglish', 'phinc'
MAX_SAMPLES = 500

def simple_cmi(query):
    tokens = re.sub(r'[^a-zA-Z\s]', '', query).lower().split()
    if len(tokens) <= 1: return 0.0
    hindi_common = {'kya', 'hai', 'hain', 'ka', 'ke', 'ki', 'ko', 'se', 'mein', 'me', 'aur', 'ya', 'nahi', 'kaise', 'karna', 'karne', 'chahiye', 'mera', 'meri', 'mujhe', 'paisa', 'paise', 'khata'}
    h_cnt = sum(1 for t in tokens if t in hindi_common)
    e_cnt = len(tokens) - h_cnt
    majority = max(h_cnt, e_cnt)
    return round((len(tokens) - majority) / len(tokens), 4)

corpus = []
queries = []

if DATASET_CHOICE == "hinglish_top":
    print("[*] Loading Google Hinglish-TOP from Hugging Face (100% Roman/Latin script)...")
    ds = load_dataset("WillHeld/hinglish_top", split="train")
    intents = {}
    for idx, row in enumerate(ds):
        if len(queries) >= MAX_SAMPLES: break
        q, intent = row.get("utterance", "").strip(), row.get("intent", "general").strip()
        cid = f"chunk_{intent}"
        if cid not in intents:
            intents[cid] = f"Official workflow, guidelines, and context information for user requests regarding {intent.replace('_', ' ')}."
            corpus.append({"chunk_id": cid, "text": intents[cid]})
        qid = f"q_{len(queries):05d}"
        queries.append({"query_id": qid, "query": q, "gold_chunk_ids": [cid], "cmi": simple_cmi(q)})

elif DATASET_CHOICE == "dolly_hinglish":
    print("[*] Loading Dolly-15k Hinglish from Hugging Face (100% Roman/Latin script)...")
    ds = load_dataset("aaditya/databricks-dolly-15k-Hinglish-Codemix", split="train")
    for idx, row in enumerate(ds):
        if len(queries) >= MAX_SAMPLES: break
        q = row.get("instruction", "").strip()
        doc = row.get("context", "").strip() or row.get("response", "").strip()
        if len(q.split()) >= 3 and len(doc.split()) >= 8:
            cid, qid = f"chunk_dolly_{len(corpus):05d}", f"q_{len(queries):05d}"
            corpus.append({"chunk_id": cid, "text": doc})
            queries.append({"query_id": qid, "query": q, "gold_chunk_ids": [cid], "cmi": simple_cmi(q)})

print(f"[✓] Successfully prepared {len(queries)} Hinglish queries over {len(corpus)} document passages.")
with open("corpus_chunks.jsonl", "w", encoding="utf-8") as f:
    for c in corpus: f.write(json.dumps(c) + "\n")
with open("queries.json", "w", encoding="utf-8") as f:
    json.dump(queries, f, indent=2)

### Step 3: Fast GPU Embedding Generation (BGE-M3)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("[*] Loading BAAI/bge-m3 on GPU...")
model = SentenceTransformer("BAAI/bge-m3")

doc_texts = [c["text"] for c in corpus]
query_texts = [q["query"] for q in queries]

print("[*] Encoding corpus chunks...")
doc_emb = model.encode(doc_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

print("[*] Encoding Hinglish queries...")
query_emb = model.encode(query_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

np.save("doc_emb_bge_m3.npy", doc_emb)
np.save("query_emb_bge_m3.npy", query_emb)
print(f"[✓] Embeddings generated successfully: doc_emb shape = {doc_emb.shape}, query_emb shape = {query_emb.shape}")

### Step 4: Run Baseline Zero-Shot Retrieval (FAISS Search)

In [ ]:
import faiss

d = doc_emb.shape[1]
index = faiss.IndexFlatIP(d)
index.add(doc_emb.astype(np.float32))

TOP_K = 10
scores, indices = index.search(query_emb.astype(np.float32), TOP_K)

mrr_list = []
hit_1_list = []
rec_10_list = []
confidence_margins = []

for i, q in enumerate(queries):
    gold = q["gold_chunk_ids"]
    ranked_cids = [corpus[idx]["chunk_id"] for idx in indices[i]]
    
    # Margin confidence (top-1 score minus top-2 score)
    conf = float(scores[i][0] - scores[i][1]) if len(scores[i]) > 1 else float(scores[i][0])
    confidence_margins.append(conf)
    
    # Rank evaluation
    rr = 0.0
    for rank, cid in enumerate(ranked_cids, 1):
        if cid in gold:
            rr = 1.0 / rank
            break
    mrr_list.append(rr)
    hit_1_list.append(1.0 if rr == 1.0 else 0.0)
    rec_10_list.append(1.0 if rr > 0.0 else 0.0)

mean_mrr = np.mean(mrr_list)
hit_at_1 = np.mean(hit_1_list)
rec_at_10 = np.mean(rec_10_list)

print("=" * 50)
print("TABLE I: RAW ZERO-SHOT RETRIEVAL PERFORMANCE (BGE-M3)")
print("=" * 50)
print(f"Queries Evaluated:   {len(queries)}")
print(f"Corpus Chunks:        {len(corpus)}")
print(f"MRR:                  {mean_mrr:.4f}")
print(f"Hit@1 (Success rate): {hit_at_1 * 100:.2f}%")
print(f"Recall@10:            {rec_at_10 * 100:.2f}%")
print(f"Mean Top-1/2 Margin:  {np.mean(confidence_margins):.4f}")
print("=" * 50)

### Step 5: Test Operators & The Overcorrection Diagnostic
We test query transformation (e.g., LQP projection / normalization) and test whether it improves failure cases or degrades already-working queries.

In [ ]:
from sklearn.linear_model import Ridge

# Split queries into Initially Correct (MRR=1) vs Initially Failed (MRR < 1)
correct_mask = np.array(hit_1_list) == 1.0
failed_mask = ~correct_mask

n_correct = int(np.sum(correct_mask))
n_failed = int(np.sum(failed_mask))

print(f"Initially Correct Queries (Rank 1): {n_correct} ({n_correct / len(queries) * 100:.1f}%)")
print(f"Initially Failed Queries:           {n_failed} ({n_failed / len(queries) * 100:.1f}%)")

# Simulate a learned projection operator (LQP)
ridge = Ridge(alpha=1.0)
X_train = query_emb[:100]
Y_train = np.array([doc_emb[corpus.index(next(c for c in corpus if c['chunk_id'] == queries[i]['gold_chunk_ids'][0]))] for i in range(100)])
ridge.fit(X_train, Y_train)

W = ridge.coef_.T
proj_emb = query_emb @ W
norms = np.linalg.norm(proj_emb, axis=1, keepdims=True)
proj_emb = np.where(norms > 0, proj_emb / norms, proj_emb)

# Search with modified queries
_, proj_indices = index.search(proj_emb.astype(np.float32), TOP_K)
proj_mrr = []
for i, q in enumerate(queries):
    gold = q["gold_chunk_ids"]
    ranked_cids = [corpus[idx]["chunk_id"] for idx in proj_indices[i]]
    rr = 0.0
    for rank, cid in enumerate(ranked_cids, 1):
        if cid in gold:
            rr = 1.0 / rank
            break
    proj_mrr.append(rr)

delta_mrr = np.array(proj_mrr) - np.array(mrr_list)
delta_correct = np.mean(delta_mrr[correct_mask]) if n_correct > 0 else 0.0
delta_failed = np.mean(delta_mrr[failed_mask]) if n_failed > 0 else 0.0

print("\n" + "=" * 50)
print("TABLE IV: OVERCORRECTION DIAGNOSIS")
print("=" * 50)
print(f"Shift on Initially Correct Queries (N={n_correct}): {delta_correct:+.4f} (Overcorrection)")
print(f"Shift on Initially Failed Queries  (N={n_failed}): {delta_failed:+.4f} (Corrective lift)")
print("=" * 50)

### Step 6: The 4-Way Reference Comparison & Oracle Gap

In [ ]:
# Strategy 1: Never Fix (Raw baseline)
mrr_never = mean_mrr

# Strategy 2: Always Fix (Unconditional operator)
mrr_always = np.mean(proj_mrr)

# Strategy 3: Simple Confidence Cutoff (Threshold Gating)
conf_thresh = np.percentile(confidence_margins, 20)  # Intervene on lowest 20% confident queries
mrr_gated = []
for i in range(len(queries)):
    if confidence_margins[i] < conf_thresh:
        mrr_gated.append(proj_mrr[i])
    else:
        mrr_gated.append(mrr_list[i])
mrr_gated = np.mean(mrr_gated)

# Strategy 4: The Empirical Oracle (Best of Raw vs Operator per query)
oracle_mrr = np.mean([max(mrr_list[i], proj_mrr[i]) for i in range(len(queries))])
oracle_gap = oracle_mrr - mrr_never

print("=" * 55)
print("THE 4-WAY REFERENCE COMPARISON & ORACLE HEADROOM GAP")
print("=" * 55)
print(f"1. Never Fix (Raw Baseline):       MRR = {mrr_never:.4f}")
print(f"2. Always Fix (Blind Pipeline):     MRR = {mrr_always:.4f}")
print(f"3. Confidence Threshold Gating:    MRR = {mrr_gated:.4f}")
print(f"4. Empirical Oracle (Upper Bound):  MRR = {oracle_mrr:.4f}")
print("-" * 55)
print(f"THE ORACLE GAP (Max headroom):      ΔMRR = {oracle_gap:+.4f}")
print("=" * 55)

### Step 7: Export & Download Experiment Results

In [ ]:
results_summary = {
    "dataset": DATASET_CHOICE,
    "n_queries": len(queries),
    "n_corpus": len(corpus),
    "mrr_never_fix": float(mrr_never),
    "mrr_always_fix": float(mrr_always),
    "mrr_confidence_gated": float(mrr_gated),
    "mrr_oracle": float(oracle_mrr),
    "oracle_gap": float(oracle_gap),
    "delta_mrr_correct": float(delta_correct),
    "delta_mrr_failed": float(delta_failed)
}

with open("experiment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=2)

!zip -r setu_colab_results.zip *.npy *.json *.jsonl

from google.colab import files
files.download("setu_colab_results.zip")
print("[✓] All results packaged and downloaded successfully!")